# Tutorial 3: Decoding the Surface Code

In Notebook 1, we decoded Shor’s code by looking up a correction for each syndrome. In Notebook 2, we encoded a logical qubit in a rotated surface-code patch with $d^2$ data qubits, performed logical operations using lattice surgery, and observed how errors produce detector events without using them to correct the logical readout.

A decoder takes the detector events and predicts a correction. A good decoder should keep the **logical failure rate** low, account for which faults are more likely under the noise model, handle errors in repeated check measurements, and run efficiently as the code grows. It does not need to identify the exact physical fault: errors that differ by a stabilizer have the same effect on the encoded information.

Why not extend Shor’s lookup table? The number of possible detector patterns grows rapidly with code size and measurement rounds. Surface-code decoders instead use the structure of the code. 

In this notebook, we use a noisy logical CNOT as a running example to show how it can be decoded. We will also build a simple matching decoder which represents possible faults as connections between detector events, assigns each connection a weight based on its likelihood, and finds a low-weight explanation for the events we observe. We will then use that explanation to adjust the logical readout and compare decoded and raw results from the same CNOT experiment.

## 1. A noisy logical CNOT

We will study a distance-three logical CNOT implemented with lattice surgery. `tqec` describes the computation and compiles it into a Stim circuit with **detectors** (comparisons of measurement results that should have even parity without faults) and **logical observables** (measurement parities that track the encoded information). We select one observable so that each shot has one logical result to decode.

For each simulated shot, Stim gives us a detector-event pattern $\mathbf{d}$ and an observable-flip bit $l$. The decoder sees $\mathbf{d}$ and predicts a correction bit $\hat l(\mathbf{d})$. A shot fails when $\hat l(\mathbf{d}) \ne l$. The raw result corresponds to always predicting zero.

We use [`tqec`'s built-in CNOT example](https://tqec.github.io/tqec/gallery/cnot.html). Its scale parameter $k=1$ gives code distance $d=2k+1=3$. `Basis.Z` chooses Z-basis preparation and readout; the two correlation surfaces track two independent logical observables. For now, we select the first.


In [1]:
from tqec import Basis, NoiseModel, compile_block_graph
from tqec.gallery.cnot import cnot

physical_error_rate = 0.001
cnot_graph = cnot(Basis.Z)
correlation_surfaces = cnot_graph.find_correlation_surfaces()
selected_observable = correlation_surfaces[0]

compiled_cnot = compile_block_graph(
    cnot_graph, observables=[selected_observable]
)
cnot_circuit = compiled_cnot.generate_stim_circuit(
    k=1,  # d = 2k + 1 = 3
    noise_model=NoiseModel.uniform_depolarizing(physical_error_rate),
)

print(f"Physical qubits: {cnot_circuit.num_qubits}")
print(f"Detectors: {cnot_circuit.num_detectors}")
print(f"Logical observables: {cnot_circuit.num_observables}")


Physical qubits: 81
Detectors: 256
Logical observables: 1
